In [2]:
import pennylane as qp
import numpy as np

In [1]:
# Circuits as functions
n_wires = 4
dev = qp.device("default.qubit", wires=n_wires)

@qp.qnode(dev)
def entangler_circuit(weights):
    qp.BasicEntanglerLayers(weights, wires=range(n_wires))
    return qp.expval(qp.PauliZ(wires=0))

print(qp.draw(entangler_circuit, level="device")([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8]]))

NameError: name 'qp' is not defined

In [7]:
# Gradient of a circuit
n_wires = 4
dev = qp.device("default.qubit", wires = n_wires)

@qp.qnode(dev)
def entangler(weights):
    qp.BasicEntanglerLayers(weights, wires = range(n_wires))
    return qp.expval(qp.PauliZ(0))

test_weights = qp.numpy.array([[0.1, 0.2, 0.3, 0.4]], requires_grad=True)
print(qp.jacobian(entangler)(test_weights))

[[ 1.38777878e-17 -1.74813749e-01 -2.66766415e-01 -3.64609810e-01]]


In [11]:
n_wires = 4
dev = qp.device("default.qubit", wires = n_wires)

@qp.qnode(dev, interface="autograd", diff_method="parameter-shift")
def entangler_fixed(diff_weights, fixed_weights):
    qp.BasicEntanglerLayers(diff_weights, wires = range(n_wires))
    qp.BasicEntanglerLayers(fixed_weights, wires = range(n_wires))
    return qp.expval(qp.PauliZ(0))

test_diff_weights = qp.numpy.array([[0.5,0.1,-0.4,0.6]], requires_grad = True)
test_fixed_weights = qp.numpy.array([[0.1,0.2,0.3,0.4]], requires_grad = False)

print(qp.jacobian(entangler_fixed)(test_diff_weights, test_fixed_weights))

[[-0.30647646  0.06160872  0.         -0.41303853]]


In [13]:
# Higher-order derivates
dev = qp.device("default.qubit", wires = 2)

@qp.qnode(dev, diff_method = "parameter-shift", max_diff = 2)
def scalar_valued_circuit(params):
  qp.RX(params[0], wires = 0)
  qp.CNOT(wires=[0,1])
  qp.RY(params[1], wires = 0)
  return qp.expval(qp.PauliZ(0))

test_params = qp.numpy.array([0.7,0.3], requires_grad = True)
qp.jacobian(qp.jacobian(scalar_valued_circuit))(test_params)

array([[-0.73068165,  0.19037934],
       [ 0.19037934, -0.73068165]])

In [17]:
# Optimizing circuits
dev = qp.device("default.qubit", wires = 2)

@qp.qnode(dev, diff_method = "parameter-shift")
def scalar_valued_circuit(params):
    qp.RX(params[0], wires = 0)
    qp.CNOT(wires=[0,1])
    qp.RY(params[1], wires = 0)
    return qp.expval(qp.PauliZ(0))

def optimize(cost_function, init_params, *steps):

    opt = qp.GradientDescentOptimizer(stepsize = 0.4)
    steps = 100
    params = init_params

    for i in range(steps):
      params = opt.step(cost_function, params)

    return params, cost_function(params)

initial_parameters = qp.numpy.array([0.7,0.3], requires_grad = True)
print(optimize(scalar_valued_circuit, initial_parameters, 100))

(tensor([3.14159265e+00, 2.86139596e-17], requires_grad=True), array(-1.))


In [22]:
# Codercise PF.4.1 - Circuits as functions
dev = qp.device("default.qubit", wires = 3)

@qp.qnode(dev)
def circuit_as_function(params):
    """
    Implements the circuit shown in the codercise statement.
    Args:
    - params (np.ndarray): [theta_0, theta_1, theta_2, theta_3]
    Returns:
    - (np.tensor): <Z0>
    """
    
    qp.StronglyEntanglingLayers(params, wires=range(3))
    return qp.expval(qp.PauliZ(wires=0))

angles = np.linspace(0, 4 * np.pi, 200)
output_values = np.array([circuit_as_function([0.5, t, 0.5, 0.5]) for t in angles])

IndexError: tuple index out of range